# TiffVolumeDataset HEIC Compression Test

This notebook tests the HEIC compression functionality of the TiffVolumeDataset class.

In [ ]:
import sys
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
import tifffile

# Add the parent directory to the path to import our modules
sys.path.append('/myhome/sdate')

from sdate.datasets import TiffVolumeDataset

## Create Synthetic Test Data

First, let's create some synthetic TIFF data for testing.

In [ ]:
# Create a test directory for synthetic data
test_data_dir = Path('/tmp/heic_test_data')
test_data_dir.mkdir(exist_ok=True)

# Generate synthetic data (smaller for faster testing)
num_frames = 50
height, width = 256, 256

print(f"Creating {num_frames} synthetic TIFF files of size {height}x{width}...")

for i in range(num_frames):
    # Create synthetic data with varying patterns
    x, y = np.meshgrid(np.linspace(0, 4*np.pi, width), np.linspace(0, 4*np.pi, height))
    data = np.sin(x + i*0.1) * np.cos(y + i*0.1) * 1000 + 2000
    
    # Add some noise
    data += np.random.normal(0, 50, data.shape)
    
    # Ensure positive values
    data = np.clip(data, 0, 4095).astype(np.uint16)
    
    # Save as TIFF
    tiff_path = test_data_dir / f"frame_{i:04d}.tiff"
    tifffile.imwrite(tiff_path, data)

print(f"Created {num_frames} TIFF files in {test_data_dir}")

## Test 1: Basic TIFF Loading (No HEIC)

First, test the basic functionality without HEIC compression.

In [ ]:
# Test basic TIFF loading
dataset_basic = TiffVolumeDataset(
    data_path=str(test_data_dir),
    volume_size=32,
    stride=16,
    num_frames=20,
    start_offset=0,
    normalize=True,
    global_normalize=True
)

print(f"Dataset size: {len(dataset_basic)}")
print(f"Volume shape: {dataset_basic.volume.shape}")

# Test loading a sub-volume
sub_volume, indices = dataset_basic[0]
print(f"Sub-volume shape: {sub_volume.shape}")
print(f"Sub-volume indices: {indices}")
print(f"Sub-volume value range: [{sub_volume.min():.3f}, {sub_volume.max():.3f}]")

## Test 2: HEIC Compression (Single Channel)

Test HEIC compression with single-channel output.

In [ ]:
# Test HEIC compression with single channel
dataset_heic = TiffVolumeDataset(
    data_path=str(test_data_dir),
    volume_size=32,
    stride=16,
    num_frames=20,
    start_offset=0,
    normalize=True,
    global_normalize=True,
    use_heic_compression=True,
    heic_quality=80,
    dual_channel=False  # Use only HEIC data
)

print(f"Dataset size: {len(dataset_heic)}")
print(f"Volume shape: {dataset_heic.volume.shape}")

# Test loading a sub-volume
sub_volume, indices = dataset_heic[0]
print(f"Sub-volume shape: {sub_volume.shape}")
print(f"Sub-volume indices: {indices}")
print(f"Sub-volume value range: [{sub_volume.min():.3f}, {sub_volume.max():.3f}]")

# Check if HEIC files were created
heic_dir = test_data_dir / 'heic_compressed'
if heic_dir.exists():
    heic_files = list(heic_dir.glob('*.heic'))
    print(f"\nCreated {len(heic_files)} HEIC files in {heic_dir}")
else:
    print("\nNo HEIC directory found")

## Test 3: HEIC Dual-Channel Loading

Test the dual-channel functionality that loads both TIFF and HEIC data.

In [ ]:
# Test dual-channel loading (TIFF + HEIC)
dataset_dual = TiffVolumeDataset(
    data_path=str(test_data_dir),
    volume_size=32,
    stride=16,
    num_frames=20,
    start_offset=0,
    normalize=True,
    global_normalize=True,
    use_heic_compression=True,
    heic_quality=80,
    dual_channel=True  # Load both TIFF and HEIC
)

print(f"Dataset size: {len(dataset_dual)}")
print(f"Volume shape: {dataset_dual.volume.shape}")

# Test loading a sub-volume
sub_volume, indices = dataset_dual[0]
print(f"Sub-volume shape: {sub_volume.shape}")
print(f"Sub-volume indices: {indices}")
print(f"Sub-volume value range: [{sub_volume.min():.3f}, {sub_volume.max():.3f}]")

# Check the two channels
tiff_channel = sub_volume[0]  # First channel (TIFF)
heic_channel = sub_volume[1]  # Second channel (HEIC)

print(f"\nTIFF channel shape: {tiff_channel.shape}")
print(f"TIFF channel range: [{tiff_channel.min():.3f}, {tiff_channel.max():.3f}]")
print(f"\nHEIC channel shape: {heic_channel.shape}")
print(f"HEIC channel range: [{heic_channel.min():.3f}, {heic_channel.max():.3f}]")

## Test 4: Visualization and Comparison

Visualize the difference between TIFF and HEIC data.

In [ ]:
# Extract a single slice for visualization
slice_idx = 15  # Middle slice

tiff_slice = tiff_channel[slice_idx].numpy()
heic_slice = heic_channel[slice_idx].numpy()

# Calculate difference
diff = np.abs(tiff_slice - heic_slice)

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# TIFF image
im1 = axes[0, 0].imshow(tiff_slice, cmap='gray')
axes[0, 0].set_title('Original TIFF')
axes[0, 0].axis('off')
plt.colorbar(im1, ax=axes[0, 0])

# HEIC image
im2 = axes[0, 1].imshow(heic_slice, cmap='gray')
axes[0, 1].set_title('HEIC Compressed')
axes[0, 1].axis('off')
plt.colorbar(im2, ax=axes[0, 1])

# Difference
im3 = axes[1, 0].imshow(diff, cmap='hot')
axes[1, 0].set_title('Absolute Difference')
axes[1, 0].axis('off')
plt.colorbar(im3, ax=axes[1, 0])

# Histogram comparison
axes[1, 1].hist(tiff_slice.flatten(), bins=50, alpha=0.7, label='TIFF', density=True)
axes[1, 1].hist(heic_slice.flatten(), bins=50, alpha=0.7, label='HEIC', density=True)
axes[1, 1].set_title('Value Distribution')
axes[1, 1].set_xlabel('Normalized Value')
axes[1, 1].set_ylabel('Density')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# Print statistics
print(f"Mean absolute difference: {diff.mean():.6f}")
print(f"Max absolute difference: {diff.max():.6f}")
print(f"Standard deviation of difference: {diff.std():.6f}")

# Calculate correlation
correlation = np.corrcoef(tiff_slice.flatten(), heic_slice.flatten())[0, 1]
print(f"Correlation coefficient: {correlation:.6f}")

## Test 5: Performance Comparison

Compare loading times and dataset iteration performance.

In [ ]:
import time

# Test iteration performance
def time_dataset_iteration(dataset, name, num_samples=10):
    print(f"\nTiming {name} dataset iteration...")
    
    start_time = time.time()
    for i in range(min(num_samples, len(dataset))):
        sub_volume, indices = dataset[i]
    end_time = time.time()
    
    avg_time = (end_time - start_time) / num_samples
    print(f"  Average time per sample: {avg_time:.4f} seconds")
    print(f"  Samples per second: {1/avg_time:.1f}")
    
    return avg_time

# Compare performance
basic_time = time_dataset_iteration(dataset_basic, "Basic TIFF")
heic_time = time_dataset_iteration(dataset_heic, "HEIC Single-Channel")
dual_time = time_dataset_iteration(dataset_dual, "HEIC Dual-Channel")

print(f"\nPerformance Summary:")
print(f"  Basic TIFF: {basic_time:.4f}s per sample")
print(f"  HEIC Single: {heic_time:.4f}s per sample ({heic_time/basic_time:.2f}x vs basic)")
print(f"  HEIC Dual: {dual_time:.4f}s per sample ({dual_time/basic_time:.2f}x vs basic)")

## Test 6: File Size Comparison

Compare the file sizes between TIFF and HEIC formats.

In [ ]:
import os

def get_directory_size(path):
    """Calculate total size of all files in a directory."""
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for filename in filenames:
            filepath = os.path.join(dirpath, filename)
            total_size += os.path.getsize(filepath)
    return total_size

def format_size(size_bytes):
    """Format size in bytes to human readable format."""
    for unit in ['B', 'KB', 'MB', 'GB']:
        if size_bytes < 1024.0:
            return f"{size_bytes:.2f} {unit}"
        size_bytes /= 1024.0
    return f"{size_bytes:.2f} TB"

# Calculate sizes
tiff_size = get_directory_size(test_data_dir)
heic_size = get_directory_size(heic_dir) if heic_dir.exists() else 0

# Count files
tiff_files = len(list(test_data_dir.glob('*.tif*')))
heic_files = len(list(heic_dir.glob('*.heic'))) if heic_dir.exists() else 0

print(f"File Counts:")
print(f"  TIFF files: {tiff_files}")
print(f"  HEIC files: {heic_files}")

print(f"\nFile Sizes:")
print(f"  TIFF directory: {format_size(tiff_size)}")
print(f"  HEIC directory: {format_size(heic_size)}")

if heic_size > 0:
    compression_ratio = tiff_size / heic_size
    print(f"  Compression ratio: {compression_ratio:.2f}x")
    print(f"  Space saved: {format_size(tiff_size - heic_size)} ({((tiff_size - heic_size) / tiff_size * 100):.1f}%)")

print(f"\nPer-file averages:")
if tiff_files > 0:
    print(f"  Average TIFF size: {format_size(tiff_size / tiff_files)}")
if heic_files > 0:
    print(f"  Average HEIC size: {format_size(heic_size / heic_files)}")

## Test 7: Real Data Test (Optional)

Test with real CT data if available.

In [ ]:
# Test with real data if available
real_data_path = Path('/myhome/data/sdate/shared/compression_paper/file_1_extracted')

if real_data_path.exists():
    print(f"Testing with real CT data from {real_data_path}")
    
    # Test with a smaller subset for speed
    dataset_real = TiffVolumeDataset(
        data_path=str(real_data_path),
        volume_size=64,
        stride=32,
        num_frames=50,  # Use fewer frames for faster testing
        start_offset=800,  # Start from middle of dataset
        normalize=True,
        global_normalize=True,
        use_heic_compression=True,
        heic_quality=85,
        dual_channel=True
    )
    
    print(f"Real dataset size: {len(dataset_real)}")
    print(f"Real volume shape: {dataset_real.volume.shape}")
    
    # Test loading a sub-volume
    sub_volume, indices = dataset_real[0]
    print(f"Real sub-volume shape: {sub_volume.shape}")
    print(f"Real sub-volume value range: [{sub_volume.min():.3f}, {sub_volume.max():.3f}]")
    
    # Quick visualization of real data
    tiff_slice = sub_volume[0, 32].numpy()  # Middle slice, TIFF channel
    heic_slice = sub_volume[1, 32].numpy()  # Middle slice, HEIC channel
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(tiff_slice, cmap='gray')
    axes[0].set_title('Real TIFF Data')
    axes[0].axis('off')
    
    axes[1].imshow(heic_slice, cmap='gray')
    axes[1].set_title('Real HEIC Data')
    axes[1].axis('off')
    
    diff = np.abs(tiff_slice - heic_slice)
    im = axes[2].imshow(diff, cmap='hot')
    axes[2].set_title('Difference')
    axes[2].axis('off')
    plt.colorbar(im, ax=axes[2])
    
    plt.tight_layout()
    plt.show()
    
    print(f"Real data compression stats:")
    print(f"  Mean absolute difference: {diff.mean():.6f}")
    print(f"  Max absolute difference: {diff.max():.6f}")
    correlation = np.corrcoef(tiff_slice.flatten(), heic_slice.flatten())[0, 1]
    print(f"  Correlation coefficient: {correlation:.6f}")
    
else:
    print(f"Real data path {real_data_path} not found. Skipping real data test.")

## Cleanup

Clean up test files.

In [ ]:
import shutil

# Uncomment the next line to clean up test data
# shutil.rmtree(test_data_dir)
# print(f"Cleaned up test data directory: {test_data_dir}")

print("Test data preserved. Uncomment the cleanup code above to remove it.")
print(f"Test data location: {test_data_dir}")
if heic_dir.exists():
    print(f"HEIC data location: {heic_dir}")